In [ ]:
# Example 1: Instantiate the Psycopg2Instrumentor and check its public API surface.
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

instrumentor = Psycopg2Instrumentor()
print(f"type: {type(instrumentor).__name__}")
print(f"mro: {[c.__name__ for c in type(instrumentor).__mro__]}")

# Should be a BaseInstrumentor with instrument / uninstrument / is_instrumented_by_opentelemetry.
for attr in ('instrument', 'uninstrument', 'instrumentation_dependencies'):
    assert hasattr(instrumentor, attr), f"missing {attr}"
    print(f"  has {attr}: {callable(getattr(instrumentor, attr))}")


In [ ]:
# Example 2: instrumentation_dependencies() reports the psycopg2 version range it supports.
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

instrumentor = Psycopg2Instrumentor()
deps = instrumentor.instrumentation_dependencies()
print(f"declared deps: {list(deps)}")
assert deps, "instrumentation_dependencies should not be empty"
# Each entry is a PEP 508 requirement string referencing psycopg2 in some form.
joined = " ".join(deps).lower()
assert "psycopg" in joined, f"expected psycopg2 mention, got: {deps}"


In [ ]:
# Example 3: instrument() + uninstrument() round-trip without an actual DB connection.
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

instrumentor = Psycopg2Instrumentor()

# Instrument patches psycopg2.connect to wrap it in a traced proxy. We don't
# need a live database to verify the instrumentor's lifecycle.
instrumentor.instrument()
print(f"after instrument(): is_instrumented_by_opentelemetry={instrumentor.is_instrumented_by_opentelemetry}")
assert instrumentor.is_instrumented_by_opentelemetry is True

instrumentor.uninstrument()
print(f"after uninstrument(): is_instrumented_by_opentelemetry={instrumentor.is_instrumented_by_opentelemetry}")
assert instrumentor.is_instrumented_by_opentelemetry is False


In [ ]:
# Example 4: After instrument(), psycopg2.connect is wrapped — verify the patch landed.
import psycopg2
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

original_connect = psycopg2.connect

instrumentor = Psycopg2Instrumentor()
instrumentor.instrument()
try:
    patched_connect = psycopg2.connect
    print(f"original psycopg2.connect: {original_connect!r}")
    print(f"patched psycopg2.connect:  {patched_connect!r}")
    assert patched_connect is not original_connect, "connect should be patched"
finally:
    instrumentor.uninstrument()

# After uninstrument, the original should be restored.
assert psycopg2.connect is original_connect, "connect should be restored"
print("connect restored after uninstrument")


In [ ]:
# Example 5: instrument() is idempotent + safely double-uninstruments.
from opentelemetry.instrumentation.psycopg2 import Psycopg2Instrumentor

instrumentor = Psycopg2Instrumentor()

instrumentor.instrument()
# Second call should be a no-op rather than raise.
instrumentor.instrument()
assert instrumentor.is_instrumented_by_opentelemetry is True

instrumentor.uninstrument()
# Second uninstrument should also be a no-op.
instrumentor.uninstrument()
assert instrumentor.is_instrumented_by_opentelemetry is False

print("Psycopg2Instrumentor instrument/uninstrument lifecycle is idempotent")
